# mT5 Fine-Tuning: Car Rental Review Summarization (DE/FR → native summary)

Fine-tunes `csebuetnlp/mT5_multilingual_XLSum` on real Floyt car rental reviews using LoRA adapters.

**Strategy:** monolingual summarization — German review → German summary, French review → French summary.
At inference time the server translates the short summary to English with DeepL (~20 words, nearly free).

This is better than mBART because mT5_multilingual_XLSum was pre-trained specifically for summarization
(XL-Sum dataset, 44 languages). mBART was pre-trained for translation, which caused hallucinations.

> **Tip:** On Kaggle go to `Session options` and select **GPU T4 x2** for faster training.

In [ ]:
!pip install -q -U torchao transformers datasets peft accelerate huggingface_hub sentencepiece

In [ ]:
import re
import time
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

## 0. Config

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME   = "csebuetnlp/mT5_multilingual_XLSum"
ADAPTER_PATH = "miladghofrani/car-rental-mt5-summarizer"

# ── Dataset ────────────────────────────────────────────────────────────────
# Native-language summaries: DE review → DE summary, FR review → FR summary.
# Created by translate_summaries_for_mt5.py from reviews_labeled_short.jsonl.
HF_DATASET = "miladghofrani/car-rental-reviews-mt5"

# ── Training ───────────────────────────────────────────────────────────────
MAX_STEPS     = None   # None = full training run
NUM_EPOCHS    = 5      # 5 epochs for better domain adaptation
LEARNING_RATE = 3e-4

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 16      # mT5-base is 580M params — rank 16 is enough for domain adaptation
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# mT5_multilingual_XLSum recommendation: normalize whitespace before tokenizing.
# Prevents tokenizer artifacts from newlines and extra spaces in raw reviews.
WHITESPACE_HANDLER = lambda k: re.sub(r'\s+', ' ', re.sub(r'\n+', ' ', k.strip()))

## 1. Detect Device

In [ ]:
def get_device():
    print('🚀 Initializing...')
    print(f'✅ PyTorch version: {torch.__version__}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'✅ Device: {device.upper()}')
    return device

device = get_device()

## 2. Load Dataset

Loads the native-language dataset from HF Hub.
Each row: `review_body` (DE/FR) → `summary` (same language as review).

> **First time only:** Run cell **2b** below on your local machine to push `reviews_labeled_mt5.jsonl` to HF Hub.

In [ ]:
def load_summarization_dataset():
    print(f"\n📥 Loading dataset from HF Hub ({HF_DATASET})...")
    raw = load_dataset(HF_DATASET, split="train")
    print(f"✅ Loaded {raw.num_rows:,} reviews")

    inputs, outputs = [], []
    for row in raw:
        body    = (row.get("review_body") or "").strip()
        summary = (row.get("summary") or "").strip()
        if not body or not summary:
            continue
        inputs.append(WHITESPACE_HANDLER(body))
        outputs.append(summary)

    split = Dataset.from_dict({"input": inputs, "output": outputs}).train_test_split(
        test_size=0.1, seed=42
    )
    print(f"✅ {split['train'].num_rows:,} train / {split['test'].num_rows:,} validation examples")
    print("\n🔍 Sample:")
    print(f"  Input  : {split['train'][0]['input'][:200]}...")
    print(f"  Output : {split['train'][0]['output']}")
    return split

dataset = load_summarization_dataset()

assert dataset["train"].num_rows > 3000, "Dataset too small — check HF_DATASET name or internet!"

import shutil, os
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/datasets"), ignore_errors=True)
print("\nCache cleared.")

### 2b. Push Dataset to HF Hub (run once from your local machine)

Run this **once from your local terminal**, not on Kaggle.
After that, the cell above always loads it from HF Hub automatically.

In [ ]:
# Run once from your LOCAL terminal:
#
#   cd /Users/milad.ghofrani/P-Projects/generative-ai
#   python3 -c "
#   from huggingface_hub import login
#   from datasets import load_dataset
#   import os
#   login(token=os.environ['HF_TOKEN'])
#   ds = load_dataset('json', data_files='data_processing/reviews_labeled_mt5.jsonl', split='train')
#   ds.push_to_hub('miladghofrani/car-rental-reviews-mt5')
#   print('Dataset pushed:', ds.num_rows, 'rows')
#   "
print("See comment above — run from local terminal, not Kaggle.")

## 3. Load Model & Tokenizer

mT5 uses the same T5 tokenizer (SentencePiece), no special language codes needed.
We load in **bfloat16** — this avoids the NaN gradient issues that fp16 caused during flan-t5 training.

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(
        f"📊 Model Parameter Report:\n"
        f"--------------------------------------------------\n"
        f"  Total Parameters:     {total:,}\n"
        f"  Trainable Parameters: {trainable:,}\n"
        f"  Percentage Trainable: {100 * trainable / total:.4f}%\n"
        f"--------------------------------------------------"
    )

print("\n🔤 Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded for {MODEL_NAME}.")

print("\n🧠 Loading mT5 model...")
# bfloat16: same dynamic range as float32, no NaN overflow risk.
# Must match the bf16=True in TrainingArguments below.
target_dtype = torch.bfloat16 if device == "cuda" else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=target_dtype).to(device)
print(f"✅ {MODEL_NAME} loaded using {target_dtype}.")
print_number_of_trainable_model_parameters(model)

## 4. Tokenize Dataset

In [ ]:
def tokenize_dataset(tokenizer, dataset):
    print("\n⚙️  Tokenizing dataset...")
    pad_id = tokenizer.pad_token_id

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input"], truncation=True, max_length=512
        )
        labels = tokenizer(
            batch["output"], truncation=True, max_length=128
        )
        model_inputs["labels"] = [
            [(t if t != pad_id else -100) for t in label]
            for label in labels["input_ids"]
        ]
        return model_inputs

    tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output"])
    print("✅ Tokenization complete.")
    return tokenized

tokenized_dataset = tokenize_dataset(tokenizer, dataset)

## 5. Inject LoRA Adapters

In [ ]:
print("\n🪄 Injecting LoRA Adapters (PEFT)...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q", "v"],   # same as T5/flan-t5 attention projection names
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)
peft_model = get_peft_model(model, lora_config)
print("✅ LoRA adapters injected.")
print_number_of_trainable_model_parameters(peft_model)

## 6. Train & Save

> Set `MAX_STEPS = None` in the Config cell for a full training run.

In [ ]:
run_dir = f"./mt5-summarizer-run-{int(time.time())}"

training_args = TrainingArguments(
    output_dir=run_dir,
    auto_find_batch_size=True,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=1,
    max_steps=MAX_STEPS if MAX_STEPS is not None else -1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,         # bfloat16 — stable dynamic range, no NaN overflow
    fp16=False,        # must be False when bf16=True
    max_grad_norm=1.0,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model, padding=True)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=collator,
)

mode = f"max_steps={MAX_STEPS}" if MAX_STEPS is not None else f"{NUM_EPOCHS} full epoch(s)"
print(f"\n🏋️  Starting mT5 summarization training ({mode})...")
trainer.train()

# Sanity-check for NaN before saving
nan_found = False
for name, param in peft_model.named_parameters():
    if 'lora' in name.lower() and torch.isnan(param).any():
        print(f"⚠️  NaN in {name}")
        nan_found = True
if nan_found:
    raise RuntimeError("Training produced NaN weights — check LEARNING_RATE or dtype mismatch.")

trainer.model.save_pretrained("./mt5-checkpoint-local")
tokenizer.save_pretrained("./mt5-checkpoint-local")
print("✅ Adapter saved to ./mt5-checkpoint-local")

## 7. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login
import os

hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN", "")

login(token=hf_token)
print("✅ Logged in to HuggingFace Hub")

In [ ]:
from huggingface_hub import HfApi

checkpoint = "./mt5-checkpoint-local"
assert os.path.isdir(checkpoint), f"Checkpoint not found at {checkpoint} — did training complete?"

api = HfApi()

# Create the repo if it doesn't exist yet (safe to call even if it already exists)
api.create_repo(repo_id=ADAPTER_PATH, repo_type="model", exist_ok=True)
print(f"✅ Repo ready: https://huggingface.co/{ADAPTER_PATH}")

api.upload_folder(
    folder_path=checkpoint,
    repo_id=ADAPTER_PATH,
    repo_type="model",
)
print(f"✅ Adapter pushed to: https://huggingface.co/{ADAPTER_PATH}")

## 8. Test Inference

Runs a German review through the fine-tuned model.
Output will be a German summary — at inference time the server translates it to English with DeepL.

In [ ]:
test_reviews = [
    # German — should produce a German summary
    (
        "Katastrophale Abwicklung. Insgesamt standen wir über eine Stunde an. "
        "Es wird versucht mit Druck Zusatzprodukte zu verkaufen — Versicherungen, anderer Wagen etc. "
        "Bei Rückgabe hat man uns zwei Schäden angelastet, die definitiv bereits vorhanden waren.",
        "DE"
    ),
    # French — should produce a French summary
    (
        "Très bonne expérience, voiture propre et personnel accueillant. "
        "La prise en charge a été rapide et sans problème.",
        "FR"
    ),
]

peft_model.eval()
for review_text, lang in test_reviews:
    inputs = tokenizer(
        WHITESPACE_HANDLER(review_text),
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        output_tokens = peft_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=80,
            num_beams=4,
            no_repeat_ngram_size=2,
            length_penalty=0.8,
        )

    summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    print(f"[{lang}] Input  : {review_text[:120]}...")
    print(f"[{lang}] Summary: {summary}")
    print(f"        (DeepL would translate this to English in production)")
    print()